---
Title: Profiling Recommendations
author:
  - name: "edesz"
    orcid: "0000-0001-9039-5496"
    attributes:
      github: "edesz"
date: 2026-05-05
language:
  title-block-published: "Last Updated"
---

# Profiling At-Risk Customers


## About

In this step, we will extract profiles about the customers predicted during inference to be at risk of canceling their credit card services at the bank.

Profiles will be created by characterizing at-risk and safe (not at-risk) customers only using the attributes in the credit card customer data provided by the client. Based on the findings, we will generate recommendations for the client to follow in order to target the at-risk customers.

As [mentioned in the scope, these recommendations will not be reported](../references/scope/07_reporting_metrics.md#limitations-of-profiling-using-raw-customer-attributes) to the client since they do not tell the client

1. whether intervention is economically justified
2. which customers should receive the most aggressive interventions
3. whether the retention campaign is expected to generate positive financial return
4. how to account for available budget

[Recommendations from estimated net savings in the previous step will be reported](./09_estimate_cohort_size_using_savings.ipynb#budget-scenario-recommendations) to the client for two budget scenarios.

:::{.callout-note}
### Outputs

Charts will be saved as `.html` files in `reports/figures`. Nothing will be exported to the R2 bucket.
:::

## Python Imports

Below we import the Python modules needed for this step

In [ ]:
#| code-fold: true
import os
from pathlib import Path

import altair as alt
import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from great_tables import GT, loc, md, style

Below are `altair` plotting settings


In [ ]:
_ = alt.data_transformers.enable("vegafusion")
_ = alt.renderers.set_embed_options(actions=False)

Define the path to the project root directory

In [ ]:
PROJ_ROOT = Path.cwd().parent

Load environment variables with secrets for use in `boto3`

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

Import the required custom modules for data loading from S3 and visualization

In [ ]:
import cc_churn.viz_altair as vzu
import r2.io_utils as r2io

## User Inputs

The variables used later are defined below

In [ ]:
columns_features_pred_proba = [
    "clientnum",
    "income_category",
    "education_level",
    "marital_status",
    "dependent_count",
    "customer_age",
    "gender",
    "card_category",
    "months_on_book",
    "num_products",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "credit_limit",
    "total_revolv_bal",
    "avg_open_to_buy",
    "total_amt_chng_q4_q1",
    "total_trans_amt",
    "total_trans_ct",
    "total_ct_chng_q4_q1",
    "avg_utilization_ratio",
    "y_pred_proba",
    "y_pred",
    "best_decision_threshold",
    "is_churned",
]

# predictions
prefix = "cloud-run"
# # predictions prefix
r2_key_pred = "all_predictions__"

We now use environment variables to define an authenticated `boto3` R2 client

In [ ]:
reports_folder = PROJ_ROOT / "reports"
figures_folder = reports_folder / "figures"

account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

## Load Data

Load churn predictions for all customers

In [ ]:
df_all_pred = r2io.pandas_read_latest_parquet_r2(
    s3_client,
    bucket_name,
    f"{prefix}/",
    r2_key_pred,
    ".parquet.gzip",
    columns_features_pred_proba,
).assign(is_at_risk=lambda df: df["y_pred"])

Use model predictions of all data to extract best decision threshold


In [ ]:
best_decision_threshold = (
    df_all_pred["best_decision_threshold"].head(1).squeeze()
)

## Exploratory Data Analysis (Post-Hoc)

### Characteristics of At-Risk Customers

Get a summary of the categorical and ordinal characteristics for all customers who are predicted to be and not to be at risk of churning

In [ ]:
df_categorical_risk_attributes = (
    pd.concat(
        [
            (
                df_all_pred.groupby([f], observed=True)
                .agg({"is_at_risk": ["sum", "count"]})
                .set_axis(["number_at_risk", "total_customers"], axis=1)
                .assign(
                    feature=f,
                    fraction_of_customers_at_risk=lambda df: (
                        df["number_at_risk"]
                        .div(df["total_customers"])
                        .mul(100)
                    ),
                )
                .reset_index()
                .rename(columns={f: "category"})
            )
            for f in [
                "card_category",
                "dependent_count",
                "education_level",
                "gender",
                "income_category",
                "marital_status",
            ]
        ]
    )
    .sort_values(
        by=["feature", "fraction_of_customers_at_risk"],
        ascending=[True, False],
    )
    .set_index(["feature", "category"])
    .reset_index()
)

This is shown below

In [ ]:
#| code-fold: true
gt = (
    GT(df_categorical_risk_attributes)
    .tab_header(md("**Summary of Categorical and Ordinal Characteristics**"))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["total_customers"]),
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns=["feature"]),
    )
    .tab_style(
        style=[style.fill(color="darkred"), style.text(color="white")],
        locations=loc.body(columns=["feature"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["fraction_of_customers_at_risk"]),
    )
    .tab_style(
        style=[
            style.fill(color="#008080"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["category"]),
    )
    .fmt_number(columns=["fraction_of_customers_at_risk"], decimals=3)
)
gt

**Observations**

From the `fraction_of_customers_at_risk` column across various categorical and ordinal features, we can make the following observations

1. Uniform Risk Distribution
   - The risk of churn is remarkably consistent across most sub-categories for all such non-numerical features. As an example, whether looking at `income_category`, `education_level`, or `marital_status`, the fraction of at-risk customers almost entirely falls between *19% and 23%* (`income_category`, `marital_status`) or *20% and 25%* (`education_level`). This suggests that demographic factors alone are not strong differentiators for churn in this specific dataset.
2. Small Outliers in Education and Card Category
   - While the distribution is mostly flat, a few groups show slightly higher risk
     - Doctorate holders have the highest risk in the education category at ~25.5%, but occur with a frequency of approximately 4.4% (451 out of approximately 10,100 customers)
     - Platinum cardholders show a risk of 30% though, again, the sample size for this group is very small (only 20 total customers)
3. Gender Consistency
   - There is only a slight difference between genders, with *Females (~21.1%)* being marginally more likely to be classified as at-risk than *Males (~20.7%)*.
4. Implication from ML Model Development
   - Because the risk is so evenly spread across these categorical features, it implies that the ML model is likely relying much more heavily on *behavioral features* (such as transaction counts, revolving balances, and contact frequency) rather than demographic profiles to make its predictions. This is not surprising and is in fact expected since we only used numerical features in the best ML model. Churn appears to be a result of *how* the customer uses the service, rather than *who* the customer is.

In summary, this suggests that churn is driven more by behavior, like transaction activity, rather than by fixed attributes like income or education.

Next, we'll get a summary of the numerical characteristics for all customers who are predicted to be and not to be at risk of canceling their credit card services at the bank

In [ ]:
df_summary_stats = (
    df_all_pred.groupby("is_at_risk")
    .agg(
        {
            "clientnum": "count",
            "customer_age": "mean",
            "months_on_book": "mean",
            "num_products": "mean",
            "months_inactive_12_mon": "mean",
            "contacts_count_12_mon": "mean",
            "credit_limit": "mean",
            "total_revolv_bal": "mean",
            "avg_open_to_buy": "mean",
            "total_amt_chng_q4_q1": "mean",
            "total_trans_amt": "mean",
            "total_ct_chng_q4_q1": "mean",
            "total_trans_ct": "mean",
            "avg_utilization_ratio": "mean",
        }
    )
    .rename(
        index={True: "At-Risk Customers", False: "Stable Customers"},
        columns={"clientnum": "num_customers"},
    )
    .transpose()
    .round(3)
    .reset_index()
    .rename(columns={"index": "Column"})
)

This is shown below

In [ ]:
#| code-fold: true
gt = (
    GT(df_summary_stats)
    .tab_header(md("**Summary of Categorical and Ordinal Characteristics**"))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["Stable Customers"]),
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns=["Column"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["At-Risk Customers"]),
    )
    .fmt_number(columns=["Stable Customers", "At-Risk Customers"], decimals=2)
)
gt

**Observations**

1. Below are the columns to focus on in charts, since they show a difference between the at-risk customers (those at risk of churning) and stable customers (those not at risk of churning)
   - `contacts_count_12_mon`
   - `total_revolv_bal`
   - `avg_utilization_ratio`
   - `total_trans_ct`
   - `total_trans_amt`

### Customer Profiling

#### Number of Credit Card Transactions

In [ ]:
#| code-fold: true
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df=df_all_pred,
    num_bins=75,
    xvar="total_trans_ct:Q",
    xtitle="Total Transaction Count",
    ytitle="Number of Customers",
    color_by_col="is_at_risk:N",
    legend_title="At Risk",
    ptitle=alt.TitleParams(
        text=(
            "At-Risk Customers Have ~40 Transactions versus ~70 for Stable "
            "Customers"
        ),
        fontSize=18,
        font="Arial",
        anchor="start",
        orient="top",
        dx=50,
        offset=10,
    ),
    y_scale="linear",
    tooltip=["total_trans_ct"],
    scale_params=dict(domain=[True, False], range=["red", "lightgrey"]),
    label_angle_box=-70,
    plot_spacing=10,
    fig_size_hist=dict(width=600, height=300),
    fig_size_bar=dict(width=75, height=300),
    save_params=dict(
        fpath=figures_folder / "fig_33_profile_total_trans_ct_histogram.html"
    ),
)
chart

At-risk customers are primarily focused in the *lower transaction count range* (35-50 transactions). This compares to stable customers who are in the 25-55 transactions range (lower) or in the 65-90 transactions (higher) range. This suggests that a drop in transaction frequency is an indicator of churn.

#### Total Dollar Value of Credit Card Transactions

In [ ]:
#| code-fold: true
chart = vzu.plot_grouped_overlapping_altair_histogram(
    df=df_all_pred,
    num_bins=75,
    xvar="total_trans_amt:Q",
    xtitle="Total Transaction Count",
    ytitle="Number of Customers",
    color_by_col="is_at_risk:N",
    legend_title="At Risk",
    ptitle=alt.TitleParams(
        text=(
            "At-Risk Customers Spend ~2,500 Dollars versus ~4,500 for Stable "
            "Customers"
        ),
        fontSize=18,
        font="Arial",
        anchor="start",
        orient="top",
        dx=50,
        offset=10,
    ),
    y_scale="linear",
    tooltip=["total_trans_amt"],
    scale_params=dict(domain=[True, False], range=["red", "lightgrey"]),
    label_angle_box=-70,
    plot_spacing=10,
    fig_size_hist=dict(width=600, height=300),
    fig_size_bar=dict(width=75, height=300),
    save_params=dict(
        fpath=figures_folder / "fig_34_profile_total_trans_amt_histogram.html"
    ),
)
chart

This is generally similar to the observations from above for transaction count.

At-risk customers are primarily focused in the *lower transaction amount range* (~2,500 dollars on average). Most stable customers spend nearly double that. The maximum spending by the predicted at-risk customers is ~10,500 dollars, but a group of stable customers spends between approximately 12,500 and 18,500 dollars. Again, this suggests that a drop in transaction value is an indicator of churn.

#### Credit Usage

In [ ]:
#| code-fold: true
chart = vzu.plot_altair_scatter_chart(
    df=df_all_pred,
    xvar="total_revolv_bal:Q",
    yvar="avg_utilization_ratio:Q",
    xtitle="Total Revolving Balance ($)",
    ytitle="Avg Utilization Ratio",
    color_by_col="is_at_risk:N",
    legend_title="At Risk",
    ptitle=alt.TitleParams(
        text="At-Risk Customers Tend to have a Low Card Usage and a Low Balance",
        fontSize=18,
        font="Arial",
        anchor="start",
        orient="top",
        dx=50,
        offset=10,
    ),
    xscale="linear",
    yscale="linear",
    scale_params=dict(domain=[True, False], range=["red", "lightgrey"]),
    fig_size=dict(width=700, height=300),
    save_params=dict(
        fpath=figures_folder
        / "fig_35_profile_tot_revol_balvs_avg_util_ratio.html"
    ),
)
chart

At-risk customers (yellow) often show nearly zero revolving balances and very low utilization ratios. This *Dormant* behavior indicates they are no longer relying on the card for their daily financial needs.

#### Customer Engagement

In [ ]:
#| code-fold: true
chart = vzu.plot_grouped_overlapping_altair_bar_chart(
    df=df_all_pred,
    xvar="contacts_count_12_mon:O",
    xtitle="Number of Contacts (Last 12 Months)",
    ytitle="Number of Customers",
    color_by_col="is_at_risk:N",
    legend_title="At Risk",
    ptitle=alt.TitleParams(
        text=(
            "At-Risk Customers are more Likely to have at least 1 Contact in the "
            "Last Year"
        ),
        fontSize=18,
        font="Arial",
        anchor="start",
        orient="top",
        dx=50,
        offset=10,
    ),
    scale_params=dict(domain=[False, True], range=["lightgrey", "darkred"]),
    y_scale="linear",
    fig_size=dict(width=700, height=300),
    save_params=dict(
        fpath=figures_folder
        / "fig_36_profile_contacts_count_12_mon_histogram.html"
    ),
)
chart

**Observations**

1. As the count moves from 0-3 to 4-6, while the absolute number of customers decreases at very high contact counts, the proportion of red to grey increases.
2. For 0-3 contacts, stable customers (grey) outnumber the at-risk customers. However, at 4+ contacts, the red bars become a much larger share of the total bar height for each category. For 6 contacts, the red portion of the bar represents a much larger percentage of that specific group compared to the lower contact groups. This indicates that these customers have a significantly higher likelihood of being at risk to cancel their credit card services at the bank. This suggests that high contact frequency is a strong predictor of churn risk. This suggests that as a customer is forced to contact the bank more frequently, likely due to unresolved issues, the probability that the ML model classifies them as at-risk increases strongly.

In summary, customers with four or more contacts in the last 12 months have a significantly higher likelihood of being classified as at-risk. Medium to high contact volume without resolution is likely a strong factor responsible for customer dissatisfaction. This suggests that unresolved issues are an important driver of credit card churn at the bank.

#### Profiles of At-Risk Customers

Customers at risk of canceling their credit card services share the following attributes

1. Dormancy
   - At-risk customers typically have significantly lower transaction counts (averaging 48 transactions compared to approximately 69 transactions for stable customers from `total_trans_ct`) and much lower revolving balances (approximately 684 dollars versus approximately 1,291 dollars, see `total_revolv_bal`). This suggests they have stopped using their credit card from this bank as their primary payment method.
2. Inactivity
   - At-risk customers show higher periods of inactivity (their `months_inactive_12_mon` is approximately 2.7 versus only 2.3 for stable customers) and lower utilization ratios (`avg_utilization_ratio` is ~0.16 versus ~0.30). This suggests a loss of engagement before the actual cancellation.
3. High Friction
   - Customers with four or more contacts in the last year are much more likely to be at risk (see the bar chart of `contacts_count_12_mon`). This indicates that unresolved issues are a driver of churn.

#### Recommendations Based on Identified Customer Profiles

Below are the targeting recommendations per customer profile

1. Dormant Users (Low Activity / Low Balance)
   - launch a [re-activation campaign](https://www.bluecore.com/blog/rethinking-retail-tackle-customer-reactivation/) with [cashback incentives](https://www.wildfire-corp.com/blog/incentives-rewards-program-participation) or point multipliers for the next few transactions to help them rebuild their card usage again
2. The Low-Value Holder (Single Product)
   - offer incentives to [bundle credit services with other bank products](https://www.suntecgroup.com/articles/bundle-up-why-banks-must-prioritize-bundling-strategies-to-retain-customers-and-grow-revenues/) (e.g. savings account linked rewards) to increase switching costs to any of the bank's competitors
3. Frustrated Customers (High Contacts)
   - perform direct outreach from a premier customer support representative to resolve outstanding issues and [offer a one-time fee waiver](https://www.theatlantic.com/ideas/archive/2025/06/customer-service-sludge/683340/)

## Conclusion

### Summary of Findings

In this notebook, we profiled the customers predicted to be at risk of canceling their credit card services. The analysis revealed three primary customer profiles and key behavioral indicators of churn

1. Transactional Dormancy
   - The at-risk customers show a significant drop in engagement, averaging only 48 transactions compared to 69 for stable customers. Their revolving balances (`total_revolv_bal`) are also roughly 50% lower (681 dollars versus 1,290 dollars), indicating the card is no longer their primary payment method.
2. Increased Friction
   - A high number of customer service contacts is a strong leading indicator of churn. Customers with four or more contacts in the last 12 months (`contacts_count_12_mon`) are significantly more likely to be classified as at-risk, likely due to unresolved service issues.
3. Inactivity and Low Utilization
   - At-risk customers exhibit longer periods of inactivity and maintain a utilization ratio of only 16% (vs. 30% for stable customers in `avg_utilization_ratio`), signaling a gradual withdrawal from the bank's services.
4. Demographic Uniformity
   - Churn risk is relatively uniform across demographic categories (income, education, marital status). It falls in the range between 19% and 23% or between 20% and 25%. This confirms that churn is driven by behavioral usage patterns rather than fixed customer attributes. This is in line with the type of features (numerical only) used in ML model development.

### Recommendations for Strategic Targeting

In order to target these at-risk customers, we recommend the bank should implement the following strategies

1. Re-activation campaigns
   - target the dormant users with cashback or point-multiplier incentives to rebuild card usage habits
2. Priority resolution
   - deploy a premier support team to reach out to frustrated customers (high contact counts) to resolve pending issues and offer retention waivers
3. Product bundling
   - increase the switching costs for low-utilization users by offering incentives to link their credit cards with other high-value bank products like savings accounts